# Figure 1I — Patient Timeline

Publication-quality multi-panel timeline showing labs, LLM confidence, medications, and line of therapy.

**Cache:** built in this notebook from the current `figures_data/figure 1/data` files (same sources as the steroid / figure 2 LOT panels). First run chunk-scans the large CSVs for this patient only; later runs reuse `figures_data/figure 1/data/.cache/<DMP_ID>.pkl` if source mtimes are unchanged. Does **not** use `~/.patient_llm_timelines_cache/`.

**LOT:** dates and ICI flags come from `OneDrive_1_8-7-2026/llm84k_ir_grade0_20260630.csv` (the current LOT skeleton). Drug names on the timeline are joined from `regimen_lot(in).csv` on start date, because the new LOT files do not carry a regimen string.


In [ ]:
import os
import re
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")

In [ ]:
# ---------------------------------------------------------------------------
# FILE PATHS & TARGET
# ---------------------------------------------------------------------------
ROOT = Path("..").resolve()
FIGURES = ROOT.parent.parent
DATA = FIGURES / "figures_data" / "figure 1" / "data"
LOT_DIR = DATA / "OneDrive_1_8-7-2026"
RESULTS = ROOT / "results"
FIG_DIR = RESULTS / "main"
FIG_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = DATA / ".cache"
if not DATA.exists():
    raise FileNotFoundError(f"Expected figure 1 data at {DATA}")
if not LOT_DIR.exists():
    raise FileNotFoundError(f"Expected LOT drop at {LOT_DIR}")
CACHE_DIR.mkdir(exist_ok=True)

TARGET_DMP_ID = 'P-0039430'
OUT_PATH = FIG_DIR / "Patient_timeline_1I.pdf"
CSV_PATH = FIG_DIR / "Patient_Timeline_1I_stats.csv"

FIG_W = 7.2
FIG_H = 3
# The layout (GridSpec height_ratios, subplots_adjust margins) is built at a
# SMALLER canvas (FIG_H / STRETCH), then the canvas is grown back to FIG_H right
# before saving. Font sizes, marker sizes, and linewidths are all specified in
# absolute points, so they don't grow with the canvas -- only the panels do. The
# net effect is every panel gets more relative room around its (fixed-size) text
# and markers, with no distortion, while the saved PDF is still exactly FIG_W x
# FIG_H. STRETCH=1 reproduces the old (undilated) behavior.
STRETCH = 1.6
FIG_H_BUILD = FIG_H / STRETCH

base_path = str(DATA)
print(f"Data: {DATA.name}")
print(f"LOT dir: {LOT_DIR.name}")
print(f"Output: {FIG_DIR.name}")


In [ ]:
# ---------------------------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------------------------
TOXICITY_COLUMNS = [
    'adrenal_insufficiency', 'colitis', 'hyperthyroidism',
    'hypothyroidism', 'pneumonitis', 'liver_toxicity'
]
SHOWCASE_LAB_ORDER = [
    'ALT', 'AST', 'Alkaline Phosphatase', 'ACTH', 'Cortisol',
    'TSH', 'T4 Free', 'T3 Free', 'Sodium', 'Potassium', 'Glucose',
    'CRP', 'LDH', 'WBC', 'Calprotectin', 'Albumin',
]
AE_TO_LABS = {
    'liver_toxicity': ['ALT', 'AST', 'Alkaline Phosphatase'],
    'hypothyroidism': ['TSH', 'T4 Free', 'T3 Free'],
    'hyperthyroidism': ['TSH', 'T4 Free', 'T3 Free'],
    'pneumonitis': ['CRP', 'LDH', 'WBC'],
    'adrenal_insufficiency': ['Cortisol', 'ACTH', 'Sodium', 'Potassium', 'Glucose'],
    'colitis': ['CRP', 'WBC', 'Calprotectin', 'Albumin'],
    'showcase': list(SHOWCASE_LAB_ORDER),
}
TOXICITY_KEYWORDS = {
    'adrenal_insufficiency': ['adrenal insufficiency', 'addison'],
    'colitis': ['colitis'],
    'hyperthyroidism': ['hyperthyroidism', 'thyrotoxicosis'],
    'hypothyroidism': ['hypothyroidism'],
    'pneumonitis': ['pneumonitis'],
    'liver_toxicity': ['hepatitis', 'liver toxicity', 'hepatotoxicity']
}
LLM_COL_MAP = {'adrenal insufficiency': 'adrenal_insufficiency', 'liver toxicity': 'liver_toxicity'}
ICI_KEYWORDS = [
    'nivolumab', 'pembrolizumab', 'ipilimumab', 'atezolizumab', 'durvalumab',
    'avelumab', 'cemiplimab', 'tremelimumab', 'dostarlimab', 'tislelizumab',
    'retifanlimab', 'botensilimab', 'opdivo', 'keytruda', 'yervoy',
    'tecentriq', 'imfinzi', 'bavencio',
]
SYSTEM_COLORS = {'liver': '#B5651D', 'thyroid': '#3E7B7E', 'adrenal': '#5E7A4F', 'gi': '#9C5B8E', 'pulmonary': '#7A6BA8'}
AE_SYSTEM = {'liver_toxicity': 'liver', 'hypothyroidism': 'thyroid', 'hyperthyroidism': 'thyroid',
             'adrenal_insufficiency': 'adrenal', 'colitis': 'gi', 'pneumonitis': 'pulmonary'}
NATURE_LAB_COLORS = {
    'ALT': SYSTEM_COLORS['liver'], 'AST': SYSTEM_COLORS['liver'], 'Alkaline Phosphatase': SYSTEM_COLORS['liver'],
    'TSH': SYSTEM_COLORS['thyroid'], 'T4 Free': SYSTEM_COLORS['thyroid'], 'T3 Free': SYSTEM_COLORS['thyroid'],
    'Cortisol': SYSTEM_COLORS['adrenal'], 'ACTH': SYSTEM_COLORS['adrenal'],
    'Sodium': SYSTEM_COLORS['adrenal'], 'Potassium': SYSTEM_COLORS['adrenal'], 'Glucose': SYSTEM_COLORS['adrenal'],
    'CRP': SYSTEM_COLORS['gi'], 'WBC': SYSTEM_COLORS['gi'], 'Calprotectin': SYSTEM_COLORS['gi'], 'Albumin': SYSTEM_COLORS['gi'],
    'LDH': SYSTEM_COLORS['pulmonary'],
}
LAB_REFERENCE_RANGES = {
    'ALT': {'low': 0, 'high': 40, 'unit': 'U/L'}, 'AST': {'low': 0, 'high': 40, 'unit': 'U/L'},
    'Alkaline Phosphatase': {'low': 40, 'high': 120, 'unit': 'U/L'},
    'TSH': {'low': 0.4, 'high': 4.5, 'unit': 'mIU/L'}, 'T4 Free': {'low': 0.8, 'high': 1.8, 'unit': 'ng/dL'},
    'T3 Free': {'low': 2.0, 'high': 4.4, 'unit': 'pg/mL'},
    'Cortisol': {'low': 6.0, 'high': 23.0, 'unit': 'mcg/dL'}, 'ACTH': {'low': 7.2, 'high': 63.0, 'unit': 'pg/mL'},
    'Sodium': {'low': 135, 'high': 145, 'unit': 'mmol/L'}, 'Potassium': {'low': 3.5, 'high': 5.0, 'unit': 'mmol/L'},
    'Glucose': {'low': 65, 'high': 110, 'unit': 'mg/dL'}, 'CRP': {'low': 0.0, 'high': 1.0, 'unit': 'mg/dL'},
    'LDH': {'low': 120, 'high': 246, 'unit': 'U/L'}, 'WBC': {'low': 4.0, 'high': 10.0, 'unit': 'K/uL'},
    'Calprotectin': {'low': 0, 'high': 50, 'unit': 'mcg/g'}, 'Albumin': {'low': 3.5, 'high': 5.0, 'unit': 'g/dL'},
}
NATURE_AXIS_COLOR = '#000000'
NATURE_GRID_COLOR = '#E0E0E0'
LOT_HIGHLIGHT_COLOR = '#B83A2A'
LOT_OTHER_COLOR = '#C9C2B5'
EMAR_SPAN_DAYS = 30
LOT_MERGE_GAP = 30
STEROID_MERGE_GAP = 30
CHUNKSIZE = 5_000_000
INVESTIGATIONAL_RE = re.compile(r'^[A-Z]{2,5}-?\d{2,}[A-Z]?\d*$', re.IGNORECASE)

# Fig 1B liver threshold is 0.01, which still draws near-zero bars.
# Skip windows well below a visible detection (this patient's first liver
# window is 0.05; later liver windows are 0.60 and 0.80).
CONF_DRAW_MIN = 0.25
LAB_YLIM_FLOOR = {'TSH': 5.0}
_DROPPED_MEDS = {'dexamethasone'}
_PLATINUM_AGENTS = {'carboplatin', 'cisplatin'}


In [ ]:
# ---------------------------------------------------------------------------
# UTILITY FUNCTIONS
# ---------------------------------------------------------------------------
def clean_columns(df):
    df.columns = (df.columns.str.replace('\ufeff', '', regex=False)
                  .str.replace('\xc3\xaf\xc2\xbb\xc2\xbf', '', regex=False).str.strip())
    return df

def find_column(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in lower_map: return lower_map[name.lower()]
    return None

def standardize_mrn(mrn_series):
    def _clean(mrn):
        try:
            s = str(mrn).strip().strip("\'\"\"").replace('P-', '').replace('p-', '').replace('MSK-', '')
            digits = re.findall(r'\d+', s)
            return str(int(digits[0])).zfill(8) if digits else None
        except (ValueError, TypeError): return None
    return mrn_series.apply(_clean)

def group_lab_names(lab_name):
    if pd.isna(lab_name) or lab_name == '': return lab_name
    lab = str(lab_name).lower().strip()
    if any(t in lab for t in ['alt', 'alanine aminotransferase', 'sgpt']): return 'ALT'
    if any(t in lab for t in ['ast', 'aspartate aminotransferase', 'sgot']): return 'AST'
    if any(t in lab for t in ['alkaline phosphatase', 'alk phos', 'alp']): return 'Alkaline Phosphatase'
    if any(t in lab for t in ['thyroid stimulating hormone', 'tsh']): return 'TSH'
    if 't4 free' in lab or 'free t4' in lab: return 'T4 Free'
    if 't3 free' in lab or 'free t3' in lab: return 'T3 Free'
    if any(t in lab for t in ['acth', 'adrenocorticotropic']): return 'ACTH'
    if any(t in lab for t in ['cortisol level', 'cortisol random', 'cortisol (']): return 'Cortisol'
    if any(t in lab for t in ['crp', 'c-reactive protein']): return 'CRP'
    if any(t in lab for t in ['ldh', 'lactate dehydrogenase']): return 'LDH'
    if 'calprotectin' in lab: return 'Calprotectin'
    if any(t in lab for t in ['wbc', 'white blood cell']): return 'WBC'
    if 'albumin' in lab: return 'Albumin'
    if any(t in lab for t in ['sodium', 'na ']): return 'Sodium'
    if any(t in lab for t in ['potassium', 'k ']): return 'Potassium'
    if 'glucose' in lab: return 'Glucose'
    return lab_name

_MED_MAP = {
    'dexmethasone': 'Dexamethasone', 'dexamethason': 'Dexamethasone',
    'dexamethasone': 'Dexamethasone', 'decadron': 'Dexamethasone',
    'prednisolone': 'Prednisone', 'prednisone': 'Prednisone',
    'methylprednisolone': 'Methylprednisolone', 'medrol': 'Methylprednisolone',
    'hydrocortisone': 'Hydrocortisone', 'solu': 'Solu-Cortef', 'solucortef': 'Solu-Cortef',
    'levothyroxine': 'Levothyroxine', 'synthroid': 'Levothyroxine',
    'liothyronine': 'Liothyronine', 'cytomel': 'Liothyronine'
}
_NON_SYSTEMIC_EXCLUSIONS = [
    'topical', 'cream', 'ointment', 'lotion', 'gel', 'foam', 'paste',
    'ophthalmic', 'eye drop', 'eye oint', 'ocular', 'ophth',
    'inhaler', 'inhalation', 'nebulizer', 'aerosol', 'spray', 'inhaled',
    'nasal', 'nose', 'intranasal', 'rectal', 'suppository', 'enema',
    'dermatologic', 'dermal', 'cutaneous', 'ear drop', 'otic',
    'transdermal', 'patch', 'sublingual', 'buccal',
    'tobradex', 'tobrex', 'maxidex', 'fml', 'lotemax',
]
def is_non_systemic_medication(med_name):
    if pd.isna(med_name) or not med_name: return False
    return any(excl in str(med_name).lower() for excl in _NON_SYSTEMIC_EXCLUSIONS)
def normalize_medication_name(med_name):
    if not med_name or med_name == 'Unknown': return 'Unknown'
    first_word = med_name.split()[0].strip('()[].,;:').lower() if med_name.split() else med_name.lower()
    for key, val in _MED_MAP.items():
        if key in first_word: return val
    return first_word.capitalize()
def merge_intervals(intervals, gap_tolerance=1):
    if not intervals: return []
    intervals = sorted(intervals, key=lambda x: x[0])
    merged = [list(intervals[0])]
    for start, end, val in intervals[1:]:
        prev = merged[-1]
        if val == prev[2] and start <= prev[1] + gap_tolerance: prev[1] = max(prev[1], end)
        else: merged.append([start, end, val])
    return [tuple(m) for m in merged]
def merge_intervals_no_value(intervals, gap_tolerance=1):
    if not intervals: return []
    intervals = sorted(intervals, key=lambda x: x[0])
    merged = [list(intervals[0])]
    for start, end in intervals[1:]:
        prev = merged[-1]
        if start <= prev[1] + gap_tolerance: prev[1] = max(prev[1], end)
        else: merged.append([start, end])
    return [tuple(m) for m in merged]
def classify_regimen_category(regimen_text):
    txt = str(regimen_text).lower()
    if any(kw in txt for kw in ICI_KEYWORDS): return 'Immuno'
    return 'Other'
def clean_regimen_name(regimen_text):
    if pd.isna(regimen_text) or not regimen_text or str(regimen_text).lower() in ('nan', 'unknown', ''): return 'Unknown'
    txt = str(regimen_text).strip()
    txt = re.sub(r'\s*\([^)]*\)', '', txt)
    parts = re.split(r'[,/+]+', txt)
    names = []
    for p in parts:
        p = p.strip()
        if not p: continue
        if INVESTIGATIONAL_RE.match(p): names.append('Investigational')
        elif p.upper() in ('5-FU', '5FU'): names.append('5-FU')
        elif '-' in p: names.append('-'.join(w.capitalize() for w in p.split('-')))
        else: names.append(p.capitalize())
    return '/'.join(names) if names else txt.capitalize()

def display_regimen_name(name):
    """Carboplatin/Cisplatin -> Platinum for display; do not merge LOT rows."""
    parts = [p.strip() for p in str(name).split('/') if p.strip()]
    out, seen_platinum = [], False
    for p in parts:
        if p.lower() in _PLATINUM_AGENTS:
            if not seen_platinum:
                out.append('Platinum')
                seen_platinum = True
        else:
            out.append(p)
    return '/'.join(out) if out else name


In [ ]:
# ---------------------------------------------------------------------------
# DATA LOADING (chunked, single-patient, with caching)
# ---------------------------------------------------------------------------
def _lot_skeleton_path(lot_dir):
    preferred = os.path.join(lot_dir, "llm84k_ir_grade0_20260630.csv")
    if os.path.exists(preferred):
        return preferred
    matches = sorted(Path(lot_dir).glob("llm84k_*_grade0_*.csv"))
    if matches:
        return str(matches[0])
    raise FileNotFoundError(f"No llm84k_*_grade0_*.csv in {lot_dir}")

def load_all_data(base_path, target_dmp_id):
    cache_dir = str(CACHE_DIR)
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = os.path.join(cache_dir, f'{target_dmp_id}.pkl')
    lot_path = _lot_skeleton_path(str(LOT_DIR))

    source_files = ['dmp_id_map.csv', 'llama_maverick_84k_patient_results.csv',
                    'llm_calls_batch_level_84k.csv', 'merged_all.csv',
                    'false_positives_analysis_zb.csv', 'comprehensive_lab_dataset_real.csv',
                    'rx_river.csv', 'emar_river.csv', 'regimen_lot(in).csv',
                    'mskimpact_clinical_data.csv', 'Patients.csv']
    current_mtimes = {f: os.path.getmtime(os.path.join(base_path, f))
                      for f in source_files if os.path.exists(os.path.join(base_path, f))}
    current_mtimes['lot_skeleton'] = os.path.getmtime(lot_path)

    if os.path.exists(cache_path):
        try:
            with open(cache_path, 'rb') as f:
                cached = pickle.load(f)
            if cached.get('mtimes') == current_mtimes:
                print("Loading from local cache")
                return cached['data']
            print("Cache invalidated (source files changed); rebuilding from figures_data")
        except Exception as exc:
            print(f"Cache read failed ({exc}); rebuilding from figures_data")

    print("Building cache from figures_data")
    data = {}

    # Resolve DMP_ID to the internal patient key
    dmp_df = pd.read_csv(os.path.join(base_path, "dmp_id_map.csv"))
    dmp_df = clean_columns(dmp_df)
    dmp_mrn_col = find_column(dmp_df, ['MRN', 'mrn', 'Patient_ID'])
    dmp_id_col = find_column(dmp_df, ['DMP_ID', 'SAMPLE_ID', 'dmp_id', 'sample_id'])
    dmp_df['MRN'] = standardize_mrn(dmp_df[dmp_mrn_col])
    target_clean = str(target_dmp_id).strip().upper()
    dmp_df['__dmp_clean'] = dmp_df[dmp_id_col].astype(str).str.strip().str.upper()
    match = dmp_df[dmp_df['__dmp_clean'] == target_clean]
    if match.empty:
        m = re.search(r'(P-?\d+)', target_clean)
        target_base = m.group(1) if m else target_clean
        dmp_df['__dmp_base'] = dmp_df['__dmp_clean'].str.extract(r'(P-?\d+)', expand=False)
        match = dmp_df[dmp_df['__dmp_base'] == target_base]
    if match.empty or pd.isna(match.iloc[0]['MRN']):
        raise ValueError(f"DMP_ID {target_dmp_id} not found")
    target_mrn = match.iloc[0]['MRN']
    target_mrn_int = int(target_mrn)
    dmp_keep = dmp_df.drop(columns=['__dmp_clean', '__dmp_base'], errors='ignore')
    data['dmp_map'] = dmp_keep[dmp_keep['MRN'] == target_mrn].copy()
    data['target_mrn'] = target_mrn
    data['target_dmp_id'] = target_dmp_id

    # Helper for chunked loading
    def chunked_load(csv_path, wanted_cols, label):
        if not os.path.exists(csv_path):
            print(f"  {label}: file not found")
            return pd.DataFrame()
        print(f"  Loading {label} (chunked)...")
        chunks = []
        for chunk in pd.read_csv(csv_path, encoding='latin-1', chunksize=CHUNKSIZE,
                                  low_memory=False,
                                  usecols=lambda c: c.replace('\ufeff', '').strip().lower() in wanted_cols):
            chunk.columns = chunk.columns.str.replace('\ufeff', '', regex=False).str.strip()
            mc = find_column(chunk, ['mrn', 'MRN', 'Patient_ID'])
            if mc is None: break
            mrn_int = pd.to_numeric(chunk[mc].astype(str).str.extract(r'(\d+)', expand=False), errors='coerce')
            mask = (mrn_int == target_mrn_int).fillna(False)
            if mask.any():
                sub = chunk.loc[mask].copy()
                sub['MRN'] = target_mrn
                chunks.append(sub)
        result = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
        print(f"    {len(result):,} records")
        return result

    # LLM
    llm_path = os.path.join(base_path, "llama_maverick_84k_patient_results.csv")
    if not os.path.exists(llm_path):
        llm_path = os.path.join(base_path, "llm_calls_batch_level_84k.csv")
    llm_wanted = {'mrn', 'patient_id', 'window_start', 'window_end',
                  'adrenal_insufficiency', 'colitis', 'hyperthyroidism',
                  'hypothyroidism', 'pneumonitis', 'liver_toxicity',
                  'adrenal insufficiency', 'liver toxicity'}
    llm_df = chunked_load(llm_path, llm_wanted, "LLM")
    if not llm_df.empty:
        llm_df['mrn'] = target_mrn
        llm_df['window_start'] = pd.to_datetime(llm_df['window_start'], errors='coerce')
        llm_df['window_end'] = pd.to_datetime(llm_df['window_end'], errors='coerce')
        llm_df = llm_df[llm_df['window_start'].notna() & llm_df['window_end'].notna()].copy()
        for old_name, new_name in LLM_COL_MAP.items():
            if old_name in llm_df.columns:
                llm_df[new_name] = pd.to_numeric(llm_df[old_name], errors='coerce')
        for tox in TOXICITY_COLUMNS:
            if tox in llm_df.columns:
                llm_df[tox] = pd.to_numeric(llm_df[tox], errors='coerce').fillna(0.0)
        llm_df = llm_df.sort_values(['mrn', 'window_start']).reset_index(drop=True)
    data['llm'] = llm_df

    # Gold standard
    merged_path = os.path.join(base_path, "merged_all.csv")
    print(f"  Loading gold standard...")
    merged_df = pd.read_csv(merged_path, encoding='latin-1')
    merged_df = clean_columns(merged_df)
    mrn_col = find_column(merged_df, ['MRN', 'mrn', 'Patient_ID'])
    if mrn_col: merged_df['mrn'] = standardize_mrn(merged_df[mrn_col])
    merged_df = merged_df[merged_df['mrn'] == target_mrn].copy()
    print(f"    {len(merged_df):,} records")
    data['merged_all'] = merged_df

    # FP corrections
    fp_path = os.path.join(base_path, "false_positives_analysis_zb.csv")
    if os.path.exists(fp_path):
        fp_df = pd.read_csv(fp_path)
        fp_df = clean_columns(fp_df)
        fp_mrn = find_column(fp_df, ['mrn', 'MRN', 'Patient_ID'])
        if fp_mrn: fp_df['mrn'] = standardize_mrn(fp_df[fp_mrn])
        fp_df = fp_df[fp_df['mrn'] == target_mrn].copy()
        data['fp_corrections'] = fp_df if not fp_df.empty else None
    else:
        data['fp_corrections'] = None

    # Lab data
    lab_wanted = {'mrn', 'patient_id', 'performed_dte', 'numeric_result', 'result_value', 'standardized_test_name'}
    lab_df = chunked_load(os.path.join(base_path, "comprehensive_lab_dataset_real.csv"), lab_wanted, "Lab")
    if not lab_df.empty:
        lab_df['PERFORMED_DTE'] = pd.to_datetime(lab_df['PERFORMED_DTE'], errors='coerce')
        if 'numeric_result' not in lab_df.columns and 'RESULT_VALUE' in lab_df.columns:
            lab_df['numeric_result'] = pd.to_numeric(lab_df['RESULT_VALUE'], errors='coerce')
        if 'standardized_test_name' in lab_df.columns:
            lab_df['standardized_test_name'] = (
                lab_df['standardized_test_name'].astype(str).str.strip().str.upper()
                .str.replace('[^A-Z0-9\\s]', '', regex=True)
                .str.replace('\\s+', ' ', regex=True))
            lab_df['grouped_lab_name'] = lab_df['standardized_test_name'].apply(group_lab_names)
    data['lab'] = lab_df if not lab_df.empty else None

    # RX
    rx_wanted = {'mrn', 'patient_id', 'drug_name', 'generic_name', 'rx_create_date', 'start_date', 'end_dt', 'discontinue_dt'}
    rx_df = chunked_load(os.path.join(base_path, "rx_river.csv"), rx_wanted, "RX")
    for col in ['RX_CREATE_DATE', 'START_DATE', 'END_DT', 'DISCONTINUE_DT']:
        if col in rx_df.columns: rx_df[col] = pd.to_datetime(rx_df[col], errors='coerce')
    data['rx'] = rx_df if not rx_df.empty else None

    # EMAR
    emar_wanted = {'mrn', 'patient_id', 'ooto_signific_dte', 'oo_ord_name'}
    emar_df = chunked_load(os.path.join(base_path, "emar_river.csv"), emar_wanted, "EMAR")
    if 'OOTO_SIGNIFIC_DTE' in emar_df.columns:
        emar_df['OOTO_SIGNIFIC_DTE'] = pd.to_datetime(emar_df['OOTO_SIGNIFIC_DTE'], errors='coerce')
    data['emar'] = emar_df if not emar_df.empty else None

    # LOT — current skeleton used by generate_steroid_cache.py and figure 2.
    # Dates / ICI flags from llm84k_*_grade0_*.csv; drug names joined from
    # regimen_lot(in).csv because the new files have no REGIMEN string.
    print(f"  Loading LOT from {Path(lot_path).name}")
    lot_df = pd.read_csv(lot_path, low_memory=False)
    lot_df = clean_columns(lot_df)
    lot_mrn_col = find_column(lot_df, ['mrn', 'MRN'])
    if lot_mrn_col is None:
        data['lot'] = None
    else:
        lot_df['MRN'] = standardize_mrn(lot_df[lot_mrn_col])
        lot_df = lot_df[lot_df['MRN'] == target_mrn].copy()
        start_col = find_column(lot_df, ['lot_start', 'APR_START_DTE'])
        end_col = find_column(lot_df, ['lot_end', 'APR_END_DTE'])
        if start_col:
            lot_df['APR_START_DTE'] = pd.to_datetime(lot_df[start_col], errors='coerce')
        if end_col:
            lot_df['APR_END_DTE'] = pd.to_datetime(lot_df[end_col], errors='coerce')
        else:
            lot_df['APR_END_DTE'] = pd.NaT
        lot_df = lot_df[lot_df['APR_START_DTE'].notna()].sort_values(['MRN', 'APR_START_DTE']).reset_index(drop=True)
        immuno_col = find_column(lot_df, ['contains_immuno', 'CONTAINS_IMMUNO'])
        chemo_col = find_column(lot_df, ['contains_chemo', 'CONTAINS_CHEMO'])
        targeted_col = find_column(lot_df, ['contains_targeted', 'CONTAINS_TARGETED'])
        lot_df['has_immuno'] = lot_df[immuno_col].fillna(False).astype(int) if immuno_col else 0
        lot_df['has_chemo'] = lot_df[chemo_col].fillna(False).astype(int) if chemo_col else 0
        lot_df['has_targeted'] = lot_df[targeted_col].fillna(False).astype(int) if targeted_col else 0
        lot_df['regimen_category'] = np.where(lot_df['has_immuno'].astype(bool), 'Immuno', 'Other')

        names_path = os.path.join(base_path, "regimen_lot(in).csv")
        lot_df['regimen_text'] = ''
        if os.path.exists(names_path) and not lot_df.empty:
            names = pd.read_csv(names_path, encoding='latin-1', low_memory=False)
            names = clean_columns(names)
            names_mrn = find_column(names, ['MRN', 'mrn'])
            names_start = find_column(names, ['APR_START_DTE', 'lot_start'])
            names_reg = find_column(names, ['REGIMEN', 'regimen', 'TREATMENT'])
            if names_mrn and names_start and names_reg:
                names['MRN'] = standardize_mrn(names[names_mrn])
                names = names[names['MRN'] == target_mrn].copy()
                names['start_key'] = pd.to_datetime(names[names_start], errors='coerce').dt.normalize()
                lot_df['start_key'] = lot_df['APR_START_DTE'].dt.normalize()
                names = names.dropna(subset=['start_key']).drop_duplicates(['MRN', 'start_key'])
                lot_df = lot_df.merge(names[['MRN', 'start_key', names_reg]], on=['MRN', 'start_key'], how='left')
                lot_df['regimen_text'] = lot_df[names_reg].fillna('').astype(str)
                lot_df = lot_df.drop(columns=['start_key', names_reg], errors='ignore')
                n_named = (lot_df['regimen_text'].str.strip() != '').sum()
                print(f"    joined regimen names for {n_named}/{len(lot_df)} LOTs")

        def _flag_label(row):
            bits = []
            if row['has_immuno']: bits.append('ICI')
            if row['has_chemo']: bits.append('chemo')
            if row['has_targeted']: bits.append('targeted')
            return '/'.join(bits) if bits else 'Unknown'

        missing_name = lot_df['regimen_text'].astype(str).str.strip().isin(['', 'nan', 'None', 'Unknown'])
        lot_df.loc[missing_name, 'regimen_text'] = lot_df.loc[missing_name].apply(_flag_label, axis=1)
        print(f"    {len(lot_df):,} records")
        data['lot'] = lot_df

    # Cancer type — prefer the LOT skeleton (same source as figure 2), then IMPACT
    cancer_map = {}
    if data.get('lot') is not None and not data['lot'].empty:
        ct_col = find_column(data['lot'], ['cancer_type_detailed', 'cancer_type'])
        if ct_col:
            val = data['lot'][ct_col].dropna()
            if not val.empty:
                cancer_map[target_mrn] = str(val.iloc[0])
    msk_path = os.path.join(base_path, "mskimpact_clinical_data.csv")
    if target_mrn not in cancer_map and os.path.exists(msk_path):
        msk = pd.read_csv(msk_path, encoding='utf-8-sig', low_memory=False)
        msk = clean_columns(msk)
        sample_col = find_column(msk, ['Sample_ID', 'Sample ID', 'SAMPLE_ID'])
        ct_col = find_column(msk, ['Cancer Type Detailed', 'CANCER_TYPE_DETAILED']) or find_column(msk, ['Cancer Type', 'CANCER_TYPE'])
        if sample_col and ct_col:
            msk['SID'] = msk[sample_col].astype(str).str.strip().str.upper()
            dmp_id_col2 = find_column(data['dmp_map'], ['SAMPLE_ID', 'DMP_ID'])
            data['dmp_map']['SID'] = data['dmp_map'][dmp_id_col2].astype(str).str.strip().str.upper()
            patient_dmp = data['dmp_map'][data['dmp_map']['MRN'] == target_mrn]
            sids = set(patient_dmp['SID'].tolist())
            msk_sub = msk[msk['SID'].isin(sids)]
            if msk_sub.empty:
                bases = patient_dmp['SID'].str.extract(r'(P-?\d+)', expand=False).dropna().tolist()
                msk['SBASE'] = msk['SID'].str.extract(r'(P-?\d+)', expand=False)
                msk_sub = msk[msk['SBASE'].isin(bases)]
            if not msk_sub.empty and pd.notna(msk_sub.iloc[0][ct_col]):
                cancer_map[target_mrn] = str(msk_sub.iloc[0][ct_col])
    if target_mrn not in cancer_map:
        patients_path = os.path.join(base_path, "Patients.csv")
        if os.path.exists(patients_path):
            pat_df = pd.read_csv(patients_path, encoding='latin-1')
            pat_df = clean_columns(pat_df)
            pat_mrn_col = find_column(pat_df, ['MRN', 'mrn', 'Patient_ID'])
            if pat_mrn_col:
                pat_df['MRN'] = standardize_mrn(pat_df[pat_mrn_col])
                pat_df = pat_df[pat_df['MRN'] == target_mrn]
                cancer_col = find_column(pat_df, ['HIST_DESC', 'Cancer Type', 'cancer_type'])
                if cancer_col and not pat_df.empty:
                    val = pat_df.iloc[0][cancer_col]
                    if pd.notna(val): cancer_map[target_mrn] = str(val)
    data['cancer_type'] = cancer_map
    print(f"  Cancer type: {cancer_map.get(target_mrn, 'Unknown')}")

    # Cache
    try:
        with open(cache_path, 'wb') as f:
            pickle.dump({'mtimes': current_mtimes, 'data': data}, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"Cached: {Path(cache_path).name}")
    except Exception as exc:
        print(f"Cache write failed ({exc})")

    return data

data = load_all_data(base_path, TARGET_DMP_ID)
target_mrn = data['target_mrn']


In [ ]:
# ---------------------------------------------------------------------------
# Gold standard & classification
# ---------------------------------------------------------------------------
def build_gold_standard(merged_all, fp_corrections):
    gs = {}
    for _, row in merged_all.iterrows():
        mrn = row.get('mrn', '')
        if pd.isna(mrn) or mrn == '': continue
        tox_text = str(row.get('Toxicity', '')).lower()
        if mrn not in gs: gs[mrn] = {tox: 0 for tox in TOXICITY_COLUMNS}
        for tox, keywords in TOXICITY_KEYWORDS.items():
            if any(kw in tox_text for kw in keywords): gs[mrn][tox] = 1
    if fp_corrections is not None:
        col_map = {f'human_score_{t}': t for t in TOXICITY_COLUMNS}
        for _, row in fp_corrections.iterrows():
            mrn = row.get('mrn')
            if pd.isna(mrn) or mrn not in gs: continue
            for h_col, tox in col_map.items():
                if h_col in fp_corrections.columns and pd.notna(row.get(h_col)):
                    if row[h_col] > 0: gs[mrn][tox] = 1
    return gs

def classify_patients(llm_df, gold_standard):
    if llm_df.empty: return {}
    patient_max = llm_df.groupby('mrn')[TOXICITY_COLUMNS].max()
    classifications = {}
    for mrn, row in patient_max.iterrows():
        if mrn not in gold_standard: continue
        classifications[mrn] = {}
        for tox in TOXICITY_COLUMNS:
            pred = 1 if row.get(tox, 0) > 0.5 else 0
            truth = gold_standard[mrn].get(tox, 0)
            if pred == 1 and truth == 1: label = 'TP'
            elif pred == 1 and truth == 0: label = 'FP'
            elif pred == 0 and truth == 1: label = 'FN'
            else: label = 'TN'
            classifications[mrn][tox] = {'class': label, 'max_conf': row.get(tox, 0.0)}
    return classifications

gold_standard = build_gold_standard(data['merged_all'], data.get('fp_corrections'))
classifications = classify_patients(data['llm'], gold_standard)
ae_dict = gold_standard.get(target_mrn, {tox: 0 for tox in TOXICITY_COLUMNS})
gs_aes = [ae for ae, v in ae_dict.items() if v == 1]
ae_classes = {ae: classifications.get(target_mrn, {}).get(ae, {}).get('class', 'FN') for ae in gs_aes}

seq_categories = []
if data.get('lot') is not None and not data['lot'].empty:
    prev_cat = None
    for _, r in data['lot'].sort_values('APR_START_DTE').iterrows():
        cat = r.get('regimen_category', 'Other')
        if cat != prev_cat: seq_categories.append(cat); prev_cat = cat
lot_sequence = ' -> '.join(seq_categories) if seq_categories else 'Unknown'
max_conf = max((classifications.get(target_mrn, {}).get(ae, {}).get('max_conf', 0.0) for ae in TOXICITY_COLUMNS), default=0.0)

print(f"AEs ({len(gs_aes)}): {', '.join(f'{a}({ae_classes[a]})' for a in gs_aes)}")
print(f"LOT: {lot_sequence} | Max conf: {max_conf:.3f}")


In [ ]:
# ---------------------------------------------------------------------------
# Data extraction & plotting functions
# ---------------------------------------------------------------------------
def compute_overall_time_range(mrn, data, ae_name):
    dates = []
    pw = data['llm'][data['llm']['mrn'] == mrn]
    if not pw.empty: dates.extend([pw['window_start'].min(), pw['window_end'].max()])
    if data.get('lab') is not None and not data['lab'].empty:
        relevant_upper = {r.upper() for r in AE_TO_LABS.get(ae_name, [])}
        plab = data['lab'][(data['lab']['MRN'] == mrn) & data['lab']['PERFORMED_DTE'].notna()]
        if not plab.empty and 'grouped_lab_name' in plab.columns:
            plab = plab[plab['grouped_lab_name'].apply(lambda x: str(x).upper() in relevant_upper if pd.notna(x) else False)]
            if not plab.empty: dates.extend([plab['PERFORMED_DTE'].min(), plab['PERFORMED_DTE'].max()])
    for src, cols in [('rx', ['START_DATE', 'RX_CREATE_DATE', 'END_DT', 'DISCONTINUE_DT']),
                      ('emar', ['OOTO_SIGNIFIC_DTE'])]:
        df = data.get(src)
        if df is not None and not df.empty:
            sub = df[df['MRN'] == mrn]
            for col in cols:
                if col in sub.columns:
                    valid = sub[col].dropna()
                    if not valid.empty: dates.extend([valid.min(), valid.max()])
    if data.get('lot') is not None and not data['lot'].empty:
        plot_d = data['lot'][data['lot']['MRN'] == mrn]
        if not plot_d.empty:
            dates.append(plot_d['APR_START_DTE'].min())
            end_v = plot_d['APR_END_DTE'].dropna()
            if not end_v.empty: dates.append(end_v.max())
    dates = [d for d in dates if pd.notna(d)]
    return (min(dates), max(dates)) if dates else (None, None)

def get_patient_labs(mrn, lab_df, relevant_labs, start_date, end_date):
    if lab_df is None or lab_df.empty: return pd.DataFrame()
    patient = lab_df[(lab_df['MRN'] == mrn) & (lab_df['PERFORMED_DTE'] >= start_date) & (lab_df['PERFORMED_DTE'] <= end_date)].copy()
    if patient.empty or 'grouped_lab_name' not in patient.columns: return pd.DataFrame()
    relevant_upper = {r.upper() for r in relevant_labs}
    patient = patient[patient['grouped_lab_name'].apply(lambda x: str(x).upper() in relevant_upper if pd.notna(x) else False)]
    return patient[patient['numeric_result'].notna()].copy()

def get_patient_medications_enhanced(mrn, rx_df, emar_df, start_date, end_date):
    total_days = (end_date - start_date).days
    all_intervals = {}
    if rx_df is not None and not rx_df.empty:
        for _, r in rx_df[rx_df['MRN'] == mrn].iterrows():
            raw_name = str(r.get('DRUG_NAME', r.get('GENERIC_NAME', 'Unknown')))[:50]
            if is_non_systemic_medication(raw_name): continue
            rx_start = r.get('START_DATE') or r.get('RX_CREATE_DATE')
            rx_end = r.get('END_DT') or r.get('DISCONTINUE_DT')
            if pd.isna(rx_start) or rx_start > end_date: continue
            if pd.notna(rx_end) and rx_end < start_date: continue
            rx_start, rx_end = max(rx_start, start_date), min(rx_end, end_date) if pd.notna(rx_end) else end_date
            med = normalize_medication_name(raw_name)
            if med.lower() in _DROPPED_MEDS:
                continue
            all_intervals.setdefault(med, []).append(((rx_start - start_date).days, (rx_end - start_date).days))
    if emar_df is not None and not emar_df.empty:
        for _, r in emar_df[emar_df['MRN'] == mrn].iterrows():
            d = r.get('OOTO_SIGNIFIC_DTE')
            if pd.isna(d) or d < start_date or d > end_date: continue
            raw_name = str(r.get('OO_ORD_NAME', 'Unknown'))[:50]
            if is_non_systemic_medication(raw_name): continue
            med = normalize_medication_name(raw_name)
            if med.lower() in _DROPPED_MEDS:
                continue
            s_day = (d - start_date).days
            all_intervals.setdefault(med, []).append((s_day, min(s_day + EMAR_SPAN_DAYS, total_days)))
    for med in all_intervals:
        all_intervals[med] = merge_intervals_no_value(all_intervals[med], gap_tolerance=STEROID_MERGE_GAP)
    return all_intervals

def get_patient_lot_enhanced(mrn, lot_df, start_date, end_date):
    if lot_df is None or lot_df.empty: return {}
    pat = lot_df[(lot_df['MRN'] == mrn) & (lot_df['APR_START_DTE'] <= end_date)].copy()
    pat = pat[(pat['APR_END_DTE'].isna()) | (pat['APR_END_DTE'] >= start_date)]
    if pat.empty: return {}
    regimens = {}
    for _, r in pat.iterrows():
        s, e = max(r['APR_START_DTE'], start_date), min(r['APR_END_DTE'], end_date) if pd.notna(r['APR_END_DTE']) else end_date
        name = clean_regimen_name(r.get('regimen_text', 'Unknown'))
        cat = r.get('regimen_category', classify_regimen_category(name))
        if name not in regimens: regimens[name] = {'intervals': [], 'category': cat}
        regimens[name]['intervals'].append(((s - start_date).days, (e - start_date).days))
    for name in regimens:
        regimens[name]['intervals'] = merge_intervals_no_value(regimens[name]['intervals'], gap_tolerance=LOT_MERGE_GAP)
    return regimens

def plot_lab_panels_enhanced(axes, lab_data, overall_start):
    if lab_data.empty or 'numeric_result' not in lab_data.columns:
        for ax in axes: ax.set_visible(False)
        return
    available = set(lab_data['grouped_lab_name'].unique())
    groups = [g for g in SHOWCASE_LAB_ORDER if g in available]
    for i, grp in enumerate(groups):
        if i >= len(axes): break
        ax = axes[i]
        sub = lab_data[lab_data['grouped_lab_name'] == grp].sort_values('PERFORMED_DTE')
        days = (sub['PERFORMED_DTE'] - overall_start).dt.days
        vals = sub['numeric_result'].values
        line_color = NATURE_LAB_COLORS.get(grp, '#2F4F6E')
        ref = LAB_REFERENCE_RANGES.get(grp)
        # Smaller marker/line -- with dense weekly-ish draws over a long timeline,
        # the original s=20/linewidth=1.5 made consecutive points overlap into a
        # solid blob regardless of panel height (a horizontal-density issue, not a
        # vertical one, which is why it survived the row-height rebalancing above).
        ax.plot(days.values, vals, color=line_color, alpha=0.8, linewidth=0.7, zorder=2, clip_on=True)
        ax.scatter(days, vals, s=6, alpha=0.85, color=line_color, zorder=3, edgecolors='none', clip_on=True)
        unit_str = f' ({ref["unit"]})' if ref else ''
        display_grp = 'Alk Phos' if grp == 'Alkaline Phosphatase' else grp
        ax.text(1.03, 0.5, f'{display_grp}{unit_str}', transform=ax.transAxes,
                fontsize=7, color=line_color, va='center', ha='left', clip_on=False)
        for sp in ax.spines.values(): sp.set_visible(False)
        # A locator can't know how short the axes actually is, so 2+ auto y-ticks
        # collide when panels are this thin. Draw one hand-placed range string
        # instead (same approach as the right-side unit label, which already
        # never overlaps because there's only one string per axis to place).
        ax.set_yticks([])
        ax.set_ylim(bottom=0)
        ymax_val = ax.get_ylim()[1]
        floor = LAB_YLIM_FLOOR.get(grp)
        if floor is not None:
            ymax_val = max(float(floor), ymax_val)
        # Keep the labeled 0–ymax range, but pad below 0 so traces (TSH on
        # 0–5 especially) do not sit on the next panel's top edge.
        pad = 0.22 * ymax_val
        ax.set_ylim(-pad, ymax_val)
        ax.text(-0.02, 0.5, f'0\u2013{ymax_val:.0f}', transform=ax.transAxes,
                fontsize=5, color=NATURE_AXIS_COLOR, va='center', ha='right', clip_on=False)
        ax.tick_params(axis='y', length=0, pad=0)
        ax.tick_params(axis='x', labelbottom=False, length=0)
    for j in range(len(groups), len(axes)): axes[j].set_visible(False)

def plot_llm_panel(ax, llm_df, mrn, overall_start, total_days):
    display_aes = list(TOXICITY_COLUMNS)
    pw = llm_df[llm_df['mrn'] == mrn].copy()
    if pw.empty:
        ax.text(0.5, 0.5, 'No LLM data', ha='center', va='center', transform=ax.transAxes, fontsize=5, style='italic', color='#555')
        for sp in ax.spines.values(): sp.set_visible(False)
        return
    for ae_idx, ae_col in enumerate(display_aes):
        if ae_col not in pw.columns: continue
        raw_intervals = []
        for _, w in pw.iterrows():
            conf = w[ae_col]
            if pd.isna(conf) or conf < CONF_DRAW_MIN: continue
            raw_intervals.append(((w['window_start'] - overall_start).days, (w['window_end'] - overall_start).days, round(conf, 2)))
        for s, e, conf in merge_intervals(raw_intervals, gap_tolerance=1):
            br, bg, bb = matplotlib.colors.to_rgb(SYSTEM_COLORS.get(AE_SYSTEM.get(ae_col, 'gi'), '#888'))
            mix = 0.25 + 0.75 * conf
            color = (1 - mix*(1-br), 1 - mix*(1-bg), 1 - mix*(1-bb))
            ax.barh(ae_idx, e-s, left=s, height=0.7, color=color, alpha=0.92, edgecolor='#CCBBAA', linewidth=0.3)
    ax.set_yticks(range(len(display_aes)))
    labels = [a.replace('_', ' ').title() for a in display_aes]
    ax.set_yticklabels(labels, fontsize=5.3)
    for tick_lbl, ae_col in zip(ax.get_yticklabels(), display_aes):
        tick_lbl.set_color(SYSTEM_COLORS.get(AE_SYSTEM.get(ae_col), NATURE_AXIS_COLOR))
    ax.invert_yaxis()
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.tick_params(left=False, labelleft=True, bottom=False, labelbottom=False)

def plot_medication_panel_enhanced(ax, med_intervals, total_days):
    if not med_intervals:
        ax.text(0.5, 0.5, 'No medication data', ha='center', va='center', transform=ax.transAxes, fontsize=5, style='italic', color='#555')
        for sp in ax.spines.values(): sp.set_visible(False)
        ax.tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)
        return
    all_meds = sorted(med_intervals.keys())
    for i, med in enumerate(all_meds):
        for s, e in med_intervals[med]:
            ax.barh(i, max(e-s, 1), left=s, height=0.5, color='#B8C9B3', alpha=0.8, edgecolor='#9AB094', linewidth=0.3)
    ax.set_yticks(range(len(all_meds)))
    ax.set_yticklabels(all_meds, fontsize=5.3, color=NATURE_AXIS_COLOR)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.tick_params(axis='y', length=0, pad=2)
    ax.tick_params(axis='x', labelbottom=False, length=0)

def plot_lot_panel_enhanced(ax, lot_regimens, total_days):
    if not lot_regimens:
        ax.text(0.5, 0.5, 'No therapy data', ha='center', va='center', transform=ax.transAxes, fontsize=5, style='italic', color='#555')
        for sp in ax.spines.values(): sp.set_visible(False)
        ax.tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)
        return
    reg_names = sorted(
        lot_regimens.keys(),
        key=lambda n: min((s for s, e in lot_regimens[n]['intervals']), default=0),
    )
    for i, name in enumerate(reg_names):
        info = lot_regimens[name]
        color = LOT_HIGHLIGHT_COLOR if info['category'] == 'Immuno' else LOT_OTHER_COLOR
        for s, e in info['intervals']:
            ax.barh(i, max(e-s, 1), left=s, height=0.55, color=color, alpha=0.8, edgecolor='none')
    ax.set_yticks(range(len(reg_names)))
    ax.set_yticklabels([display_regimen_name(n) for n in reg_names], fontsize=5.3)
    for tick_lbl, name in zip(ax.get_yticklabels(), reg_names):
        tick_lbl.set_color(LOT_HIGHLIGHT_COLOR if lot_regimens[name]['category'] == 'Immuno' else NATURE_AXIS_COLOR)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.tick_params(axis='y', length=0, pad=2)
    ax.tick_params(axis='x', labelbottom=False, length=0)

In [ ]:
# ---------------------------------------------------------------------------
# Generate the timeline
# ---------------------------------------------------------------------------
ae_name = 'showcase'
overall_start, overall_end = compute_overall_time_range(target_mrn, data, ae_name)
total_days = (overall_end - overall_start).days
print(f"Timeline: {total_days} days")

lab_data = get_patient_labs(target_mrn, data.get('lab'), AE_TO_LABS[ae_name], overall_start, overall_end)
med_intervals = get_patient_medications_enhanced(target_mrn, data.get('rx'), data.get('emar'), overall_start, overall_end)
lot_regimens = get_patient_lot_enhanced(target_mrn, data.get('lot'), overall_start, overall_end)

available = set(lab_data['grouped_lab_name'].unique()) if not lab_data.empty and 'grouped_lab_name' in lab_data.columns else set()
lab_groups = [g for g in SHOWCASE_LAB_ORDER if g in available]

n_lab_panels = max(len(lab_groups), 1)
n_llm_rows = 6
# Lab traces need vertical range (TSH on 0–5 was flattening into the
# baseline). Bar rows only need room for one label, so they can be shorter.
# Canvas size is unchanged: GridSpec ratios only.
per_lab_h = 1.65
ROW_H = 0.62
enh_llm_h = max(0.7, n_llm_rows * ROW_H)
enh_med_h = max(0.8, max(len(med_intervals), 1) * ROW_H)
enh_lot_h = max(0.8, max(len(lot_regimens), 1) * ROW_H)

n_panels = n_lab_panels + 3
heights = [per_lab_h] * n_lab_panels + [enh_llm_h, enh_med_h, enh_lot_h]

print(f"Panels: {n_lab_panels} labs + LLM + meds + LOT")
print(f"Labs: {lab_groups}")
print(f"Meds: {list(med_intervals.keys())}")
print(f"LOT: {list(lot_regimens.keys())}")

# ---------------------------------------------------------------------------
# Summary CSV (no patient identifiers)
# ---------------------------------------------------------------------------
def _iso(ts):
    if ts is None or pd.isna(ts):
        return ''
    return pd.Timestamp(ts).date().isoformat()

def _day(ts):
    if ts is None or pd.isna(ts):
        return ''
    return int((pd.Timestamp(ts) - overall_start).days)

rows = []
cancer_type = data.get('cancer_type', {}).get(target_mrn, 'Unknown')
rows.append({
    'panel': '1I', 'section': 'patient', 'item': 'example_patient',
    'gold_standard': '', 'llm_class': '', 'max_confidence': '',
    'start_date': _iso(overall_start), 'end_date': _iso(overall_end),
    'start_day': 0, 'end_day': total_days,
    'category': cancer_type, 'n': '', 'min_value': '', 'median_value': '',
    'max_value': '', 'unit': '',
})

for tox in TOXICITY_COLUMNS:
    info = classifications.get(target_mrn, {}).get(tox, {})
    gold = int(ae_dict.get(tox, 0))
    rows.append({
        'panel': '1I', 'section': 'ae', 'item': tox,
        'gold_standard': gold, 'llm_class': info.get('class', 'TN' if gold == 0 else 'FN'),
        'max_confidence': round(float(info.get('max_conf', 0.0)), 4),
        'start_date': '', 'end_date': '', 'start_day': '', 'end_day': '',
        'category': '', 'n': '', 'min_value': '', 'median_value': '',
        'max_value': '', 'unit': '',
    })

lot_src = data.get('lot')
if lot_src is not None and not lot_src.empty:
    for _, r in lot_src.sort_values('APR_START_DTE').iterrows():
        rows.append({
            'panel': '1I', 'section': 'lot',
            'item': display_regimen_name(clean_regimen_name(r.get('regimen_text', 'Unknown'))),
            'gold_standard': '', 'llm_class': '', 'max_confidence': '',
            'start_date': _iso(r.get('APR_START_DTE')), 'end_date': _iso(r.get('APR_END_DTE')),
            'start_day': _day(r.get('APR_START_DTE')), 'end_day': _day(r.get('APR_END_DTE')),
            'category': r.get('regimen_category', ''),
            'n': '', 'min_value': '', 'median_value': '', 'max_value': '', 'unit': '',
        })

for med, intervals in med_intervals.items():
    for s, e in intervals:
        rows.append({
            'panel': '1I', 'section': 'medication', 'item': med,
            'gold_standard': '', 'llm_class': '', 'max_confidence': '',
            'start_date': _iso(overall_start + pd.Timedelta(days=int(s))),
            'end_date': _iso(overall_start + pd.Timedelta(days=int(e))),
            'start_day': int(s), 'end_day': int(e),
            'category': '', 'n': '', 'min_value': '', 'median_value': '',
            'max_value': '', 'unit': '',
        })

if not lab_data.empty and 'grouped_lab_name' in lab_data.columns:
    for grp in lab_groups:
        sub = lab_data[lab_data['grouped_lab_name'] == grp]
        vals = pd.to_numeric(sub['numeric_result'], errors='coerce').dropna()
        ref = LAB_REFERENCE_RANGES.get(grp, {})
        rows.append({
            'panel': '1I', 'section': 'lab', 'item': grp,
            'gold_standard': '', 'llm_class': '', 'max_confidence': '',
            'start_date': _iso(sub['PERFORMED_DTE'].min()) if 'PERFORMED_DTE' in sub.columns else '',
            'end_date': _iso(sub['PERFORMED_DTE'].max()) if 'PERFORMED_DTE' in sub.columns else '',
            'start_day': _day(sub['PERFORMED_DTE'].min()) if 'PERFORMED_DTE' in sub.columns else '',
            'end_day': _day(sub['PERFORMED_DTE'].max()) if 'PERFORMED_DTE' in sub.columns else '',
            'category': '', 'n': int(len(vals)),
            'min_value': round(float(vals.min()), 4) if len(vals) else '',
            'median_value': round(float(vals.median()), 4) if len(vals) else '',
            'max_value': round(float(vals.max()), 4) if len(vals) else '',
            'unit': ref.get('unit', ''),
        })

summary_df = pd.DataFrame(rows)
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(CSV_PATH, index=False)
print(f"Saved CSV: {CSV_PATH.name}")


In [ ]:

# ---------------------------------------------------------------------------
# Render and save
# ---------------------------------------------------------------------------

cancer_type = data.get('cancer_type', {}).get(target_mrn, 'Unknown')
ae_strs = [f"{a.replace('_', ' ')}({ae_classes.get(a, '?')})" for a in gs_aes]

fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor='white')
gs_layout = GridSpec(n_panels, 1, height_ratios=heights, hspace=0.015)

# ---------------------------------------------------------------------------
# Lab panels
# ---------------------------------------------------------------------------

lab_axes = [fig.add_subplot(gs_layout[i]) for i in range(n_lab_panels)]
plot_lab_panels_enhanced(lab_axes, lab_data, overall_start)

# ---------------------------------------------------------------------------
# LLM panel
# ---------------------------------------------------------------------------

ax_llm = fig.add_subplot(gs_layout[n_lab_panels])
plot_llm_panel(
    ax_llm,
    data['llm'],
    target_mrn,
    overall_start,
    total_days
)

# ---------------------------------------------------------------------------
# Medication panel
# ---------------------------------------------------------------------------

ax_med = fig.add_subplot(gs_layout[n_lab_panels + 1])
plot_medication_panel_enhanced(
    ax_med,
    med_intervals,
    total_days
)

# ---------------------------------------------------------------------------
# LOT panel
# ---------------------------------------------------------------------------

ax_lot = fig.add_subplot(gs_layout[n_lab_panels + 2])
plot_lot_panel_enhanced(
    ax_lot,
    lot_regimens,
    total_days
)

# ---------------------------------------------------------------------------
# Shared x-axis
# ---------------------------------------------------------------------------

all_axes = lab_axes + [ax_llm, ax_med, ax_lot]

for ax in all_axes:
    if ax.get_visible():
        ax.set_xlim(0, total_days)

ax_lot.tick_params(
    axis='x',
    labelbottom=True,
    labelsize=6,
    colors=NATURE_AXIS_COLOR,
    length=2,
    width=0.3,
    pad=2
)

ax_lot.set_xlabel(
    'Days',
    fontsize=7,
    color=NATURE_AXIS_COLOR,
    labelpad=2
)

# ---------------------------------------------------------------------------
# Layout
#
# Keep the right edge at 0.80 so the plot does not move toward the
# confidence bar. Move the left edge from 0.36 -> 0.25 so the
# timeline becomes wider primarily toward the left.
# ---------------------------------------------------------------------------

fig.subplots_adjust(
    left=0.25,
    right=0.80,
    bottom=0.045,
    top=0.985
)

# ---------------------------------------------------------------------------
# Section dividers / grid lines
#
# IMPORTANT: These are created AFTER subplots_adjust() so that their
# positions match the FINAL subplot positions.
# ---------------------------------------------------------------------------

visible_axes = [ax for ax in all_axes if ax.get_visible()]

if visible_axes:
    # Determine the actual left/right edges of the plot area.
    plot_left = min(ax.get_position().x0 for ax in visible_axes)
    plot_right = max(ax.get_position().x1 for ax in visible_axes)

    # Major section dividers:
    # LLM / Medication / LOT boundaries
    for idx in [n_lab_panels, n_lab_panels + 1, n_lab_panels + 2]:
        if idx < len(all_axes) and all_axes[idx].get_visible():
            bbox = all_axes[idx].get_position()

            fig.add_artist(
                plt.Line2D(
                    [plot_left, plot_right],
                    [bbox.y1 + 0.003, bbox.y1 + 0.003],
                    transform=fig.transFigure,
                    color='#D8D8D8',
                    linewidth=0.35,
                    clip_on=False
                )
            )

    # Subtle dividers between individual lab panels
    for lab_idx in range(1, n_lab_panels):
        if lab_axes[lab_idx].get_visible():
            bbox = lab_axes[lab_idx].get_position()

            fig.add_artist(
                plt.Line2D(
                    [plot_left, plot_right],
                    [bbox.y1 + 0.003, bbox.y1 + 0.003],
                    transform=fig.transFigure,
                    color='#ECECEC',
                    linewidth=0.2,
                    clip_on=False
                )
            )

# ---------------------------------------------------------------------------
# Confidence legend bar
# ---------------------------------------------------------------------------

conf_y_offset = -0.05

llm_bbox = ax_llm.get_position()

legend_ax = fig.add_axes([
    0.96,
    llm_bbox.y0 + conf_y_offset,
    0.012,
    llm_bbox.height
])

grad = np.linspace(0.0, 1.0, 256).reshape(-1, 1)

legend_ax.imshow(
    grad,
    aspect='auto',
    cmap='Greys',
    origin='lower',
    extent=[0, 1, 0, 1],
    vmin=0,
    vmax=1
)

legend_ax.set_xticks([])
legend_ax.set_yticks([0.0, 0.5, 1.0])
legend_ax.set_yticklabels(['0.0', '0.5', '1.0'], fontsize=6)
legend_ax.yaxis.tick_right()

legend_ax.tick_params(
    axis='y',
    length=2,
    pad=2,
    color=NATURE_AXIS_COLOR
)

for sp in legend_ax.spines.values():
    sp.set_visible(False)

fig.text(
    0.966,
    llm_bbox.y1 + conf_y_offset + 0.006,
    'Conf.',
    fontsize=5,
    ha='center',
    va='bottom',
    color=NATURE_AXIS_COLOR
)

# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------

with PdfPages(str(OUT_PATH)) as pdf:
    pdf.savefig(
        fig,
        dpi=300,
        facecolor='white',
        bbox_inches='tight'
    )

print(f"Saved: {OUT_PATH.name}")
plt.show()



In [ ]:
# ---------------------------------------------------------------------------
# Render and save
# ---------------------------------------------------------------------------

cancer_type = data.get('cancer_type', {}).get(target_mrn, 'Unknown')
ae_strs = [f"{a.replace('_', ' ')}({ae_classes.get(a, '?')})" for a in gs_aes]

fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor='white')
gs_layout = GridSpec(n_panels, 1, height_ratios=heights, hspace=0.015)

# ---------------------------------------------------------------------------
# Lab panels
# ---------------------------------------------------------------------------

lab_axes = [fig.add_subplot(gs_layout[i]) for i in range(n_lab_panels)]
plot_lab_panels_enhanced(lab_axes, lab_data, overall_start)

# ---------------------------------------------------------------------------
# LLM panel
# ---------------------------------------------------------------------------

ax_llm = fig.add_subplot(gs_layout[n_lab_panels])
plot_llm_panel(
    ax_llm,
    data['llm'],
    target_mrn,
    overall_start,
    total_days
)

# ---------------------------------------------------------------------------
# Medication panel
# ---------------------------------------------------------------------------

ax_med = fig.add_subplot(gs_layout[n_lab_panels + 1])
plot_medication_panel_enhanced(
    ax_med,
    med_intervals,
    total_days
)

# ---------------------------------------------------------------------------
# LOT panel
# ---------------------------------------------------------------------------

ax_lot = fig.add_subplot(gs_layout[n_lab_panels + 2])
plot_lot_panel_enhanced(
    ax_lot,
    lot_regimens,
    total_days
)

# ---------------------------------------------------------------------------
# Shared x-axis
# ---------------------------------------------------------------------------

all_axes = lab_axes + [ax_llm, ax_med, ax_lot]

for ax in all_axes:
    if ax.get_visible():
        ax.set_xlim(0, total_days)

ax_lot.tick_params(
    axis='x',
    labelbottom=True,
    labelsize=6,
    colors=NATURE_AXIS_COLOR,
    length=2,
    width=0.3,
    pad=2
)

ax_lot.set_xlabel(
    'Days',
    fontsize=7,
    color=NATURE_AXIS_COLOR,
    labelpad=2
)

# ---------------------------------------------------------------------------
# Layout
#
# Widen the timeline in BOTH directions.
#
# left=0.20  -> more space gained on the left
# right=0.85 -> more space gained on the right
#
# The confidence bar remains at x=0.96, leaving separation between the
# plot/labels and the confidence legend.
# ---------------------------------------------------------------------------

fig.subplots_adjust(
    left=0.20,
    right=0.85,
    bottom=0.045,
    top=0.985
)

# ---------------------------------------------------------------------------
# Section dividers / grid lines
#
# These are created AFTER subplots_adjust() so that their positions match
# the final subplot boundaries exactly.
# ---------------------------------------------------------------------------

visible_axes = [ax for ax in all_axes if ax.get_visible()]

if visible_axes:

    # Actual left and right boundaries of the plotting region
    plot_left = min(
        ax.get_position().x0
        for ax in visible_axes
    )

    plot_right = max(
        ax.get_position().x1
        for ax in visible_axes
    )

    # -----------------------------------------------------------------------
    # Major section dividers
    # -----------------------------------------------------------------------

    for idx in [
        n_lab_panels,
        n_lab_panels + 1,
        n_lab_panels + 2
    ]:

        if idx < len(all_axes) and all_axes[idx].get_visible():

            bbox = all_axes[idx].get_position()

            fig.add_artist(
                plt.Line2D(
                    [plot_left, plot_right],
                    [bbox.y1 + 0.003, bbox.y1 + 0.003],
                    transform=fig.transFigure,
                    color='#D8D8D8',
                    linewidth=0.35,
                    clip_on=False
                )
            )

    # -----------------------------------------------------------------------
    # Subtle dividers between individual lab panels
    # -----------------------------------------------------------------------

    for lab_idx in range(1, n_lab_panels):

        if lab_axes[lab_idx].get_visible():

            bbox = lab_axes[lab_idx].get_position()

            fig.add_artist(
                plt.Line2D(
                    [plot_left, plot_right],
                    [bbox.y1 + 0.003, bbox.y1 + 0.003],
                    transform=fig.transFigure,
                    color='#ECECEC',
                    linewidth=0.2,
                    clip_on=False
                )
            )

# ---------------------------------------------------------------------------
# Confidence legend bar
# ---------------------------------------------------------------------------

conf_y_offset = -0.05

llm_bbox = ax_llm.get_position()

legend_ax = fig.add_axes([
    0.96,
    llm_bbox.y0 + conf_y_offset,
    0.012,
    llm_bbox.height
])

grad = np.linspace(0.0, 1.0, 256).reshape(-1, 1)

legend_ax.imshow(
    grad,
    aspect='auto',
    cmap='Greys',
    origin='lower',
    extent=[0, 1, 0, 1],
    vmin=0,
    vmax=1
)

legend_ax.set_xticks([])
legend_ax.set_yticks([0.0, 0.5, 1.0])
legend_ax.set_yticklabels(
    ['0.0', '0.5', '1.0'],
    fontsize=6
)

legend_ax.yaxis.tick_right()

legend_ax.tick_params(
    axis='y',
    length=2,
    pad=2,
    color=NATURE_AXIS_COLOR
)

for sp in legend_ax.spines.values():
    sp.set_visible(False)

fig.text(
    0.966,
    llm_bbox.y1 + conf_y_offset + 0.006,
    'Conf.',
    fontsize=5,
    ha='center',
    va='bottom',
    color=NATURE_AXIS_COLOR
)

# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------

with PdfPages(str(OUT_PATH)) as pdf:
    pdf.savefig(
        fig,
        dpi=300,
        facecolor='white',
        bbox_inches='tight'
    )

print(f"Saved: {OUT_PATH.name}")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Render and save
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Consistent figure typography
# ---------------------------------------------------------------------------

FONT_FAMILY = 'Arial'
TICK_FONT_SIZE = 6
AXIS_LABEL_SIZE = 7
TEXT_FONT_SIZE = 6

plt.rcParams.update({
    'font.family': FONT_FAMILY,
    'font.size': TEXT_FONT_SIZE,
    'axes.labelsize': AXIS_LABEL_SIZE,
    'xtick.labelsize': TICK_FONT_SIZE,
    'ytick.labelsize': TICK_FONT_SIZE,
})

cancer_type = data.get('cancer_type', {}).get(target_mrn, 'Unknown')
ae_strs = [f"{a.replace('_', ' ')}({ae_classes.get(a, '?')})" for a in gs_aes]

fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor='white')

gs_layout = GridSpec(
    n_panels,
    1,
    height_ratios=heights,
    hspace=0.015
)

# ---------------------------------------------------------------------------
# Lab panels
# ---------------------------------------------------------------------------

lab_axes = [
    fig.add_subplot(gs_layout[i])
    for i in range(n_lab_panels)
]

plot_lab_panels_enhanced(
    lab_axes,
    lab_data,
    overall_start
)

# ---------------------------------------------------------------------------
# LLM panel
# ---------------------------------------------------------------------------

ax_llm = fig.add_subplot(gs_layout[n_lab_panels])

plot_llm_panel(
    ax_llm,
    data['llm'],
    target_mrn,
    overall_start,
    total_days
)

# ---------------------------------------------------------------------------
# Medication panel
# ---------------------------------------------------------------------------

ax_med = fig.add_subplot(gs_layout[n_lab_panels + 1])

plot_medication_panel_enhanced(
    ax_med,
    med_intervals,
    total_days
)

# ---------------------------------------------------------------------------
# LOT panel
# ---------------------------------------------------------------------------

ax_lot = fig.add_subplot(gs_layout[n_lab_panels + 2])

plot_lot_panel_enhanced(
    ax_lot,
    lot_regimens,
    total_days
)

# ---------------------------------------------------------------------------
# Shared x-axis
# ---------------------------------------------------------------------------

all_axes = lab_axes + [
    ax_llm,
    ax_med,
    ax_lot
]

for ax in all_axes:
    if ax.get_visible():

        # Same x-axis range for every panel
        ax.set_xlim(0, total_days)

        # ---------------------------------------------------------------
        # Standardize tick appearance
        # ---------------------------------------------------------------

        ax.tick_params(
            axis='both',
            labelsize=TICK_FONT_SIZE,
            colors=NATURE_AXIS_COLOR,
            length=2,
            width=0.3,
            pad=2
        )

        # Force tick-label font family and size
        for label in ax.get_xticklabels():
            label.set_fontfamily(FONT_FAMILY)
            label.set_fontsize(TICK_FONT_SIZE)

        for label in ax.get_yticklabels():
            label.set_fontfamily(FONT_FAMILY)
            label.set_fontsize(TICK_FONT_SIZE)

        # ---------------------------------------------------------------
        # Standardize axis labels
        # ---------------------------------------------------------------

        ax.xaxis.label.set_fontfamily(FONT_FAMILY)
        ax.xaxis.label.set_fontsize(AXIS_LABEL_SIZE)

        ax.yaxis.label.set_fontfamily(FONT_FAMILY)
        ax.yaxis.label.set_fontsize(AXIS_LABEL_SIZE)

        # ---------------------------------------------------------------
        # Standardize text drawn directly inside the axes
        # ---------------------------------------------------------------

        for text in ax.texts:
            text.set_fontfamily(FONT_FAMILY)
            text.set_fontsize(TEXT_FONT_SIZE)


# ---------------------------------------------------------------------------
# Bottom x-axis
# ---------------------------------------------------------------------------

ax_lot.tick_params(
    axis='x',
    labelbottom=True,
    labelsize=TICK_FONT_SIZE,
    colors=NATURE_AXIS_COLOR,
    length=2,
    width=0.3,
    pad=2
)

for label in ax_lot.get_xticklabels():
    label.set_fontfamily(FONT_FAMILY)
    label.set_fontsize(TICK_FONT_SIZE)

ax_lot.set_xlabel(
    'Days',
    fontsize=AXIS_LABEL_SIZE,
    color=NATURE_AXIS_COLOR,
    labelpad=2,
    fontfamily=FONT_FAMILY
)

# ---------------------------------------------------------------------------
# Layout
#
# IMPORTANT:
# The plot keeps essentially the SAME WIDTH as before:
#
# Previous: left=0.25, right=0.80  -> width = 0.55
# New:      left=0.29, right=0.84  -> width = 0.55
#
# Therefore we are NOT making the plot larger.
# We are simply shifting the entire timeline to the RIGHT.
#
# This uses the remaining white space on the right while keeping the
# confidence bar safely separated.
# ---------------------------------------------------------------------------

fig.subplots_adjust(
    left=0.29,
    right=0.90,
    bottom=0.045,
    top=0.985
)

# ---------------------------------------------------------------------------
# Section dividers / grid lines
#
# Created AFTER subplots_adjust() so that they exactly match the final
# plotting area.
# ---------------------------------------------------------------------------

visible_axes = [
    ax for ax in all_axes
    if ax.get_visible()
]

if visible_axes:

    # Actual left/right boundaries of the plotting region
    plot_left = min(
        ax.get_position().x0
        for ax in visible_axes
    )

    plot_right = max(
        ax.get_position().x1
        for ax in visible_axes
    )

    # ---------------------------------------------------------------
    # Major section dividers
    # ---------------------------------------------------------------

    for idx in [
        n_lab_panels,
        n_lab_panels + 1,
        n_lab_panels + 2
    ]:

        if idx < len(all_axes) and all_axes[idx].get_visible():

            bbox = all_axes[idx].get_position()

            fig.add_artist(
                plt.Line2D(
                    [plot_left, plot_right],
                    [
                        bbox.y1 + 0.003,
                        bbox.y1 + 0.003
                    ],
                    transform=fig.transFigure,
                    color='#D8D8D8',
                    linewidth=0.35,
                    clip_on=False
                )
            )

    # ---------------------------------------------------------------
    # Subtle dividers between lab panels
    # ---------------------------------------------------------------

    for lab_idx in range(1, n_lab_panels):

        if lab_axes[lab_idx].get_visible():

            bbox = lab_axes[lab_idx].get_position()

            fig.add_artist(
                plt.Line2D(
                    [plot_left, plot_right],
                    [
                        bbox.y1 + 0.003,
                        bbox.y1 + 0.003
                    ],
                    transform=fig.transFigure,
                    color='#ECECEC',
                    linewidth=0.2,
                    clip_on=False
                )
            )

# ---------------------------------------------------------------------------
# Confidence legend bar
# ---------------------------------------------------------------------------

conf_y_offset = -0.05

llm_bbox = ax_llm.get_position()

legend_ax = fig.add_axes([
    0.96,
    llm_bbox.y0 + conf_y_offset,
    0.012,
    llm_bbox.height
])

grad = np.linspace(
    0.0,
    1.0,
    256
).reshape(-1, 1)

legend_ax.imshow(
    grad,
    aspect='auto',
    cmap='Greys',
    origin='lower',
    extent=[0, 1, 0, 1],
    vmin=0,
    vmax=1
)

legend_ax.set_xticks([])

legend_ax.set_yticks([
    0.0,
    0.5,
    1.0
])

legend_ax.set_yticklabels(
    ['0.0', '0.5', '1.0'],
    fontsize=TICK_FONT_SIZE,
    fontfamily=FONT_FAMILY
)

legend_ax.yaxis.tick_right()

legend_ax.tick_params(
    axis='y',
    length=2,
    pad=2,
    color=NATURE_AXIS_COLOR,
    labelsize=TICK_FONT_SIZE
)

for label in legend_ax.get_yticklabels():
    label.set_fontfamily(FONT_FAMILY)
    label.set_fontsize(TICK_FONT_SIZE)

for sp in legend_ax.spines.values():
    sp.set_visible(False)

# Conf. title
fig.text(
    0.966,
    llm_bbox.y1 + conf_y_offset + 0.006,
    'Conf.',
    fontsize=TICK_FONT_SIZE,
    fontfamily=FONT_FAMILY,
    ha='center',
    va='bottom',
    color=NATURE_AXIS_COLOR
)

# ---------------------------------------------------------------------------
# Save
#
# Compute the tight bounding box ONCE and reuse it for every output format.
# If each savefig() call is left to recompute bbox_inches='tight' on its
# own, the PDF backend and the PNG (Agg) backend can measure text/tick
# extents slightly differently, so the "tight" crop -- and therefore the
# final physical size -- can end up a bit different between the two files.
# Using one explicit Bbox object guarantees the PDF and PNG are identical
# in size.
# ---------------------------------------------------------------------------

fig.canvas.draw()
tight_bbox = fig.get_tightbbox(fig.canvas.get_renderer()).padded(0.02)

with PdfPages(str(OUT_PATH)) as pdf:

    pdf.savefig(
        fig,
        dpi=300,
        facecolor='white',
        bbox_inches=tight_bbox
    )

print(f"Saved: {OUT_PATH.name}")
print(f"Figure size (tight bbox): {tight_bbox.width:.4f} in x {tight_bbox.height:.4f} in")

plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# PNG version -- identical physical dimensions to the PDF saved above.
# Reuses tight_bbox from the previous cell instead of the string 'tight',
# so the crop matches exactly (see note in the previous cell).
# ---------------------------------------------------------------------------

OUT_PATH_PNG = OUT_PATH.with_suffix('.png')

fig.savefig(
    OUT_PATH_PNG,
    dpi=600,                # high-res; bump to 900-1200 if you need print quality
    facecolor='white',
    bbox_inches=tight_bbox  # same Bbox object used for the PDF -- guarantees matching size
)

print(f"Saved: {OUT_PATH_PNG.name}")
print(f"Figure size (tight bbox): {tight_bbox.width:.4f} in x {tight_bbox.height:.4f} in")

plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# GRANT VERSION — taller, no text overlap, dimensions don't need to match Nature spec
# ---------------------------------------------------------------------------
GRANT_OUT_PATH = RESULTS / "extra_timeline" / "Patient_timeline_1I_grant.pdf"
GRANT_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

GRANT_FIG_W = 9.5   # a bit wider too, to give the right-side lab labels more room
GRANT_FIG_H = 7.0   # roughly 2.3x taller than the Nature version (3in -> 7in)

fig_grant = plt.figure(figsize=(GRANT_FIG_W, GRANT_FIG_H), facecolor='white')
gs_layout_grant = GridSpec(n_panels, 1, height_ratios=heights, hspace=0.15, figure=fig_grant)

# Lab panels
lab_axes_grant = [fig_grant.add_subplot(gs_layout_grant[i]) for i in range(n_lab_panels)]
plot_lab_panels_enhanced(lab_axes_grant, lab_data, overall_start)

# LLM panel
ax_llm_grant = fig_grant.add_subplot(gs_layout_grant[n_lab_panels])
plot_llm_panel(ax_llm_grant, data['llm'], target_mrn, overall_start, total_days)

# Medication panel
ax_med_grant = fig_grant.add_subplot(gs_layout_grant[n_lab_panels + 1])
plot_medication_panel_enhanced(ax_med_grant, med_intervals, total_days)

# LOT panel
ax_lot_grant = fig_grant.add_subplot(gs_layout_grant[n_lab_panels + 2])
plot_lot_panel_enhanced(ax_lot_grant, lot_regimens, total_days)

# Shared x-axis
all_axes_grant = lab_axes_grant + [ax_llm_grant, ax_med_grant, ax_lot_grant]
for ax in all_axes_grant:
    if ax.get_visible(): ax.set_xlim(0, total_days)

ax_lot_grant.tick_params(axis='x', labelbottom=True, labelsize=8, colors=NATURE_AXIS_COLOR, length=2, width=0.3, pad=2)
ax_lot_grant.set_xlabel('Days', fontsize=9, color=NATURE_AXIS_COLOR, labelpad=4)

# Section dividers
for idx in [n_lab_panels, n_lab_panels + 1, n_lab_panels + 2]:
    if idx < len(all_axes_grant):
        bbox = all_axes_grant[idx].get_position()
        fig_grant.add_artist(plt.Line2D([0.04, 0.94], [bbox.y1 + 0.003, bbox.y1 + 0.003],
                       transform=fig_grant.transFigure, color='#D8D8D8', linewidth=0.35, clip_on=False))
for lab_idx in range(1, n_lab_panels):
    if lab_axes_grant[lab_idx].get_visible():
        bbox = lab_axes_grant[lab_idx].get_position()
        fig_grant.add_artist(plt.Line2D([0.04, 0.94], [bbox.y1 + 0.003, bbox.y1 + 0.003],
                       transform=fig_grant.transFigure, color='#ECECEC', linewidth=0.2, clip_on=False))

# More generous right margin than the Nature version, since the taller figure
# gives lab-label text more vertical room to avoid overlapping neighboring rows
plt.tight_layout(rect=[0.28, 0.05, 0.82, 0.92])

# Confidence legend bar
conf_y_offset = -0.03
llm_bbox = ax_llm_grant.get_position()
legend_ax_grant = fig_grant.add_axes([0.96, llm_bbox.y0 + conf_y_offset, 0.012, llm_bbox.height])
grad = np.linspace(0.0, 1.0, 256).reshape(-1, 1)
legend_ax_grant.imshow(grad, aspect='auto', cmap='Greys', origin='lower', extent=[0, 1, 0, 1], vmin=0, vmax=1)
legend_ax_grant.set_xticks([])
legend_ax_grant.set_yticks([0.0, 0.5, 1.0])
legend_ax_grant.set_yticklabels(['0.0', '0.5', '1.0'], fontsize=7)
legend_ax_grant.yaxis.tick_right()
legend_ax_grant.tick_params(axis='y', length=2, pad=2, color=NATURE_AXIS_COLOR)
for sp in legend_ax_grant.spines.values(): sp.set_visible(False)
fig_grant.text(0.966, llm_bbox.y1 + conf_y_offset + 0.006, 'Conf.', fontsize=6,
         ha='center', va='bottom', color=NATURE_AXIS_COLOR)

# Save as a separate file — original Nature-spec PDF is untouched
with PdfPages(str(GRANT_OUT_PATH)) as pdf:
    pdf.savefig(fig_grant, dpi=300, facecolor='white', bbox_inches='tight')
print(f"Saved grant version: {GRANT_OUT_PATH.name}")
plt.show()